# 🎯 [K14_M04] 요약 수치 직접 확인하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K14/k14_m04_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·부대관리 (Level 1) / K14 군 생활 데이터 분석 |
| **미션** | 미션 04. 요약 수치 직접 확인하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 군 생활 데이터 분석, HITL 수기 검산, 시각화 왜곡 방지 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [Colab 환경 설정 및 헬퍼 함수] 데이터 분석, 가상 데이터 로더 & 마크다운 렌더러
# ============================================================================
!pip install -q pandas numpy tabulate matplotlib seaborn

import pandas as pd
import numpy as np
import json
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# 폰트 설정 (Colab / Linux 호환)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# ----------------------------------------------------------------------------
# [데이터셋 자동 로드] 가상 한빛대대 4주간 생활관 생활 데이터 (12행)
# ----------------------------------------------------------------------------
data_records = [
    {'주차': '1주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 112, '싸지방': 54, 'PX': 140, '수면만족': 3.8, '식단만족': 3.9, '건의': 3},
    {'주차': '1주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 98, '싸지방': 31, 'PX': 160, '수면만족': 3.6, '식단만족': 3.5, '건의': 4},
    {'주차': '1주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 104, '싸지방': 47, 'PX': 118, '수면만족': 3.7, '식단만족': 3.2, '건의': 5},
    {'주차': '2주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 118, '싸지방': 58, 'PX': 138, '수면만족': 3.8, '식단만족': 3.8, '건의': 2},
    {'주차': '2주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 92, '싸지방': 29, 'PX': 166, '수면만족': 3.4, '식단만족': 3.4, '건의': 5},
    {'주차': '2주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 106, '싸지방': 50, 'PX': 121, '수면만족': 3.6, '식단만족': 3.1, '건의': 7},
    {'주차': '3주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 121, '싸지방': 61, 'PX': 142, '수면만족': 3.7, '식단만족': 4.0, '건의': 4},
    {'주차': '3주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 85, '싸지방': 26, 'PX': 171, '수면만족': 3.2, '식단만족': 3.4, '건의': 7},
    {'주차': '3주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 103, '싸지방': 49, 'PX': 117, '수면만족': 3.7, '식단만족': 3.3, '건의': 9},
    {'주차': '4주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 125, '싸지방': 66, 'PX': 135, '수면만족': 3.7, '식단만족': 4.1, '건의': 3},
    {'주차': '4주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 80, '싸지방': 24, 'PX': 178, '수면만족': 3.0, '식단만족': 3.3, '건의': 8},
    {'주차': '4주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 108, '싸지방': 52, 'PX': 120, '수면만족': 3.7, '식단만족': 3.2, '건의': 12}
]

df = pd.DataFrame(data_records)

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드합니다."""
    filename = f"k14_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K14", "../uploads/K14"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 분석 결과 및 검증 리포트를 시각적으로 렌더링합니다."""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 분석 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ [K14] 실습 환경, 12행 생활관 데이터셋(df) 및 시각화 렌더러 준비 완료!")
display(df)



---

## 🎯 미션 04. 요약 수치 직접 확인하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **AI가 계산한 수치를 직접 확인합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **AI가 알려 준 합계·평균·증감률을 계산기나 표 계산 도구로 다시 계산해 맞음·틀림으로 검토하고 핵심 정보를 식별하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `수치 검산표, 오류 원인과 수정 요청` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 4 수치 검산표

| **항목·중대** | **AI가 알려 준 값** | **직접 계산한 값** | **맞음·틀림** | **오류 원인** |
| --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] AI 결과 검증 — AI가 계산한 수치를 직접 다시 계산해 오류를 찾고 바로잡았는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(수치 검산표, 오류 원인과 수정 요청)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 04.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_04_prompt = """[AI 산출 수치 1:1 수기 검산 및 CoT 오류 수정 피드백 질의]
당신은 데이터 팩트체크 및 HITL(Human-in-the-Loop) 검증 전문가입니다. 앞서 AI가 산출한 4주간 합계 수치에 대해 원문 데이터를 바탕으로 1:1 수기 검산을 수행하고 판정표(PASS/FAIL)를 작성해 주십시오.
특히 계산 착오가 발생하기 쉬운 항목(예: 2중대 PX 이용 합계 665 오산 등)을 식별하여 오류 원인을 분석하고, 생각의 사슬(CoT, Chain-of-Thought) 기법을 적용하여 올바른 수치(675)를 도출하도록 단계별 덧셈 과정을 유도하는 피드백 프롬프트를 작성해 주십시오."""

mission_04_result = """## 1. 1:1 수기 검산 및 판정표
| 항목 및 대상 | 원문 데이터 (1~4주차) | AI 제시값 | 직접 검산값 | 판정 (PASS/FAIL) | 오류 원인 분석 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 1중대 체력 합계 | 112 + 118 + 121 + 125 | 476 | 476 | [x] PASS | 일치 |
| 2중대 체력 합계 | 98 + 92 + 85 + 80 | 355 | 355 | [x] PASS | 일치 |
| 3중대 체력 합계 | 104 + 106 + 103 + 108 | 421 | 421 | [x] PASS | 일치 |
| 1중대 싸지방 합계 | 54 + 58 + 61 + 66 | 239 | 239 | [x] PASS | 일치 |
| 2중대 싸지방 합계 | 31 + 29 + 26 + 24 | 110 | 110 | [x] PASS | 일치 |
| 3중대 싸지방 합계 | 47 + 50 + 49 + 52 | 198 | 198 | [x] PASS | 일치 |
| 2중대 PX 합계 | 160 + 166 + 171 + 178 | 665 (예시 오류) | 675 | [x] FAIL | 178을 168로 단순 오산 |
| 3중대 건의 합계 | 5 + 7 + 9 + 12 | 33 | 33 | [x] PASS | 일치 |

## 2. CoT 오류 수정 요청 프롬프트 및 교정본
- **피드백 프롬프트**:
  ```text
  2중대 4주간 PX 이용 합계에서 계산 착오가 발생했습니다. 
  1주차(160), 2주차(166), 3주차(171), 4주차(178) 수치를 하나씩 순서대로 더하는 과정(CoT)을 단계별로 보여주고 최종 올바른 합계를 도출해 주세요.
  ```
- **AI 교정 응답**:
  - 1단계: 160 + 166 = 326
  - 2단계: 326 + 171 = 497
  - 3단계: 497 + 178 = 675 (정정 완료)

## 3. 도전 과제: 재검산 일관성 비교
- **새 세션 재검산 결과**: 
- **HITL 검증의 중요성 메모**:"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 04. 요약 수치 직접 확인하기", mission_04_prompt, mission_04_result)



In [ ]:
# [미션 04 실습 보조] AI 산출 수치 직접 1:1 검산 (HITL 연산 검증)
print('=== 중대별 4주간 총합 (체련실/싸지방/PX/건의) ===')
totals = df.groupby('중대')[['체력단련실', '싸지방', 'PX', '건의']].sum()
display(totals)

print('\n=== 중대별 4주간 만족도 평균 (수면/식단) ===')
averages = df.groupby('중대')[['수면만족', '식단만족']].mean().round(2)
display(averages)

# AI 요약 결과와 대조하여 1:1 맞음/틀림을 판정하세요.